# Ellipsoids node regression tasks {-}

This notebook contains an interactive workflow for the ellipsoids node regression experiments (the random bandlimited signal and individual eigenfunction learning tasks). Use it to (1) generate the ellipsoids sample datasets, (2) run k-folds cross-validation for MFCN and baseline models, and (3) summarize results in tables and plots, for both tasks.

> **Important**: to set up directories for the project based on a string key for the system in use, set the `MACHINE` keys (e.g., "desktop") and `ROOT` path (to the project folder on your "desktop" machine) args in the `__post_init__` method of `args_template.py`. Then, use your chosen `MACHINE` key when initializing `args` in this first cell and in script calls with the `--machine` argument.

In [ ]:
import sys
sys.path.insert(0, '../')
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import results_utilities as ru
import ellipsoids_node_regress.args as a

args = a.Args(MACHINE="desktop")

## Dataset creation {-}

### 10 linear combos of 1...$\kappa$ eigenvectors signals {-}

> If you wish to run both noiseless and noisy experiments, you'll need to generate both datasets: one with the `--add_noise` flag and one without.

Also: 
- Make sure you have set the correct paths for each `--machine` flag in `args_template.py`.
- You may have to provide the absolute path to `create_melanoma_pyg_dataset.py` in the cell below.
- Make sure `python3` is calling the correct version of python (>=3.11), and that it is available as an ipykernel to Jupyter.

In [ ]:
# noiseless
! python3 "create_ellip_node_data.py" \
--machine "desktop" \
-n 1024 -a 3 -b 2 -c 1 \
--ambient_dim 8 \
--type combination \
--n_datasets 10 \
--min_evec_i 1 \
--max_evec_i 20

In [ ]:
# with noise
! python3 "create_ellip_node_data.py" \
--machine "desktop" \
-n 1024 -a 3 -b 2 -c 1 \
--ambient_dim 8 \
--type combination \
--n_datasets 10 \
--min_evec_i 1 \
--max_evec_i 20 \
--add_noise

### Single eigenvectors signals 1...$\kappa$ {-}

In [ ]:
! python3 "create_ellip_node_data.py" \
--machine "desktop" \
-n 1024 -a 3 -b 2 -c 1 \
--ambient_dim 8 \
--type single \
--min_evec_i 1 \
--max_evec_i 20 \
--evec_stride 1

## Model training (k-folds cross-validation) {-}

**Include model flags in the command-line arguments for the model types you wish to fit:**

- MFCN-low-pass-spectral filter model: set the value of $t$ as the `SPECTRAL_C` argument in ellipsoids_node_regress/args.py file. Flag: `--mcn_spectral`

- MFCN-low-pass-approx (diffusion) filter model: `--mcn_p`

- MFCN with spectral wavelets: `--mfcn_spectral`

- MFCN with dyadic diffusion (approx) wavelets: `--mfcn_p`

- MFCN with 'Infogain' diffusion (approx) wavelets: `--mfcn_p --smart_p_wavelets` [not used in ellipsoids tasks in original paper]

- Baseline GNNs: `--gcn --gat --sage --gin`

### Eigenvector combinations {-}

#### Runs {-}

- You may have to provide the absolute path to `experiments_cv.py` in the cell below.
- You may have to modify the `--multi_dataset_dir` argument to the full path to the random bandlimited signals dataset on your machine.

In [ ]:
! python3.11 "../train_scripts/experiments_cv.py" \
--machine desktop \
--mfcn_spectral \
--spectral_c 1.0 \
--mcn_within_filter_chan_out 32,16 \
--mfcn_within_filter_chan_out 8,4 \
--mfcn_cross_filter_combos_out 8,4 \
--dataset ellipsoids-combination \
--multi_dataset_dir "{PATH_TO_DATA}/ellipsoids_node_1_1024_knn-auto_evecs1-20_combinations_ambient8_noisy" \
--n_folds 5 \
--n_epochs 10000 \
--burn_in 250 \
--patience 50 \
--learn_rate 0.01 \
--batch_size 1024 \
--verbosity 0

In [ ]:
import os
show_noisy = False
n_datasets = 10
root = "{ROOT_DIR}"
# args.MODEL_SAVE_DIR = {NEW_DIR}  # if needed


model_dirs = (
    ""
)
model_suffix_dict = {
    "" : ""
}

results_dfs = [None] * n_datasets
for i in range(n_datasets):
    filter_tuple = ('dataset', f'comb_{i}')
    results_dfs[i] = ru.get_cv_results_df(
        args, 
        model_dirs, 
        model_suffix_dict=model_suffix_dict,
        include_times=True,
        filter_tuple=filter_tuple,
        decimal_round=None
    )

df = ru.avg_avg_results_df(
    pd.concat(results_dfs),
    sort_col_tuple=('R2', 'mean', 'nanmean'),
    sort_col_ascends=[False]
)

# set decimal rounding settings by (multilevel) cols
col_rounds = {
    ('R2', 'mean', 'nanmean'): 2,
    ('R2', 'std', 'mean_std'): 2,
    ('mse', 'mean', 'nanmean'): 4,
    ('mse', 'std', 'mean_std'): 4,
    ('sec_per_epoch', 'mean', 'nanmean'): 4,
    ('sec_per_epoch', 'std', 'mean_std'): 4,
    ('n_epochs', 'mean', 'nanmean'): 0,
    ('n_epochs', 'std', 'mean_std'): 0,
    ('train_min_per_fold', 'mean', 'nanmean'): 2,
    ('train_min_per_fold', 'std', 'mean_std'): 2
}
final_colnames = {
    'R2': '$R^2$',
    'mse': 'MSE',
    'sec_per_epoch': 'sec/epoch',
    'n_epochs': 'n epochs',
    'train_min_per_fold': 'train min/fold'
}

df, df_latex = ru.get_mean_pm_std_df(
    df=df,
    mean_std_colnames=('R2', 'mse', 'sec_per_epoch', 'n_epochs', 'train_min_per_fold'),
    col_rounds=col_rounds,
    final_colnames=final_colnames,
    mean_subcol_tuple=('mean', 'nanmean'),
    std_subcol_tuple=('std', 'mean_std'),
    sort_col_tuple=('R2', 'mean', 'nanmean'),
    sort_col_ascends=[False],
    best_bolding_fns_dict=None
)

# inspect
print(df_latex)
df

#### Ablation study

In [ ]:
! python3.11 "../train_scripts/experiments_cv.py" \
--machine desktop \
--mcn_spectral \
--mcn_within_filter_chan_out 32,16 \
--mfcn_within_filter_chan_out None \
--mfcn_cross_filter_combos_out None \
--dataset ellipsoids-combination \
--multi_dataset_dir "{PATH_TO_DATA}/ellipsoids_node/ellipsoids_node_1_1024_knn-auto_evecs1-20_combinations_ambient8_noisy" \
--n_folds 5 \
--n_epochs 10000 \
--burn_in 250 \
--patience 50 \
--learn_rate 0.01 \
--batch_size 1024 \
--verbosity 0

#### Results table {-}

After running cross-validations on the desired models, add the paths to their results folders to the `model_dirs: Tuple[str]` variables in this block (for convenience, separated by whether the model was fit on noisy data or not--adjust the `show_noisy: bool` for this also):

In [ ]:
show_noisy = True
n_datasets = 10
# args.MODEL_SAVE_DIR = "{NEW_DIR}"
# args.RESULTS_SAVE_DIR = "{NEW_DIR}"

# NOTE: spectral_c = 0.5 for all models here
if show_noisy:
    ###  W/ NOISE
    model_dirs = (
        ""
    )
    model_suffix_dict = {
        "" : ""
    }

else: 
    ### NOISELESS
    model_dirs = (
        "",
    )
    model_suffix_dict = {
        "" : ""
    }

results_dfs = [None] * n_datasets
for i in range(n_datasets):
    filter_tuple = ('dataset', f'comb_{i}')
    results_dfs[i] = ru.get_cv_results_df(
        args, 
        model_dirs, 
        model_suffix_dict=model_suffix_dict,
        include_times=True,
        filter_tuple=filter_tuple,
        decimal_round=None
    )

df = ru.avg_avg_results_df(
    pd.concat(results_dfs),
    sort_col_tuple=('R2', 'mean', 'nanmean'),
    sort_col_ascends=[False]
)

# set decimal rounding settings by (multilevel) cols
col_rounds = {
    ('R2', 'mean', 'nanmean'): 2,
    ('R2', 'std', 'mean_std'): 2,
    ('mse', 'mean', 'nanmean'): 4,
    ('mse', 'std', 'mean_std'): 4,
    ('sec_per_epoch', 'mean', 'nanmean'): 4,
    ('sec_per_epoch', 'std', 'mean_std'): 4,
    ('n_epochs', 'mean', 'nanmean'): 0,
    ('n_epochs', 'std', 'mean_std'): 0,
    ('train_min_per_fold', 'mean', 'nanmean'): 2,
    ('train_min_per_fold', 'std', 'mean_std'): 2
}
final_colnames = {
    'R2': '$R^2$',
    'mse': 'MSE',
    'sec_per_epoch': 'sec/epoch',
    'n_epochs': 'n epochs',
    'train_min_per_fold': 'train min/fold'
}

df, df_latex = ru.get_mean_pm_std_df(
    df=df,
    mean_std_colnames=('R2', 'mse', 'sec_per_epoch', 'n_epochs', 'train_min_per_fold'),
    col_rounds=col_rounds,
    # final_modelnames=final_modelnames,
    final_colnames=final_colnames,
    mean_subcol_tuple=('mean', 'nanmean'),
    std_subcol_tuple=('std', 'mean_std'),
    sort_col_tuple=('R2', 'mean', 'nanmean'),
    sort_col_ascends=[False],
    best_bolding_fns_dict=None
)

# inspect
print(df_latex) 
df

### Single eigenvectors {-}

#### Run experiments {-}

> You may have to modify the `--multi_dataset_dir` argument to the full path to the single eigenvectors dataset on your machine.

In [ ]:
! python3.11 "../train_scripts/experiments_cv.py" \
--machine desktop \
--mcn_spectral \
--spectral_c 1.0 \
--mcn_within_filter_chan_out 32,16 \
--mfcn_within_filter_chan_out 8,4 \
--mfcn_cross_filter_combos_out 8,4 \
--dataset ellipsoids-single \
--multi_dataset_dir "{PATH_TO_DATA}/ellipsoids_node/ellipsoids_node_1_1024_knn-auto_single_evecs1-20_ambient8" \
--n_folds 5 \
--n_epochs 10000 \
--burn_in 250 \
--patience 50 \
--learn_rate 0.01 \
--batch_size 1024 \
--verbosity 0

#### Results {-}

After running cross-validations on the desired models, add the paths to their results folders to the `model_dirs: Tuple[str]` variable in this block:

In [ ]:
import os
root = "{ROOT_DIR}"
# args.MODEL_SAVE_DIR = {NEW_DIR}  # if needed
# args.RESULTS_SAVE_DIR = {NEW_DIR}  # if needed

model_dirs = (
    ""
)
model_suffix_dict = {
    "" : ""
}

evec_keys = [f"evec_{i+1}" for i in range(0, 20, 1)]
sort_keys = [('mse', 'mean'), ('mse', 'std'), ('R2', 'mean'), ('R2', 'std'), 'model']
sort_asc_bools = [True, True, False, True, True]

# construct individual model results dataframes
dfs = [None] * len(evec_keys)
for i, evec_key in enumerate(evec_keys):
    results_df = ru.get_cv_results_df(
        args, 
        model_dirs, 
        model_suffix_dict=model_suffix_dict,
        include_times=False,
        filter_tuple=('dataset', evec_key),
        decimal_round=None,
    )
    # append column identifying single eigenvector that was fit
    # add 1 to each eigenvector so indexes are 2, ..., 21 (first 20 nontrivial)
    results_df['evec'] = int(evec_key.split("_")[1]) + 1
    dfs[i] = results_df
    # print(f"\n{evec_key}\n", results_df)

# concatenate individual model results into one dataframe
df = pd.concat(dfs)

# numerically, some R^2s could be < 0
# -> enforce 0 floor for R^2 (and 0 for st devs where R^2 = 0)
df.loc[df[('R2', 'mean')] < 0., ('R2', 'std')] = 0.
df_num = df._get_numeric_data()
df_num[df_num <= 0] = 0

# sort models within evec groups, by (R^2 [descending], MSE [ascend.]) performance
# ties within metrics are broken by smaller standard deviation
# any remaining ties are broken by alphabetical model name
df = df.groupby(('evec')) \
    .apply(lambda x: x.sort_values(sort_keys, ascending=sort_asc_bools)) \
    .droplevel('evec')

# inspect results
df.head(10)

#### Summary table of results {-}

In [ ]:
# reload(ru)

# set decimal rounding settings by (multilevel) cols
col_rounds = {
    ('R2', 'mean'): 4,
    ('R2', 'std'): 4,
    ('mse', 'mean'): 4,
    ('mse', 'std'): 4,
    ('sec_per_epoch', 'mean'): 4,
    ('sec_per_epoch', 'std'): 4,
    ('n_epochs', 'mean'): 0,
    ('n_epochs', 'std'): 0,
    ('train_min_per_fold', 'mean'): 2,
    ('train_min_per_fold', 'std'): 2
}

# generate final formatted table and its latex string
full_df_format, full_df_latex = ru.get_mean_pm_std_df(
    df=df,
    mean_std_colnames=('R2', 'mse'), #'sec_per_epoch', 'n_epochs', 'train_min_per_fold'),
    col_rounds=col_rounds,
    preserve_colnames=('evec', ),
    final_colnames={'evec': 'Eigenfunction', 'R2': '$R^2$', 'mse': 'MSE'}, 
    best_bolding_fns_dict=None,
    mean_subcol_tuple=('mean', ),
    std_subcol_tuple=('std', ),
    sort_col_tuple=[('evec',),('R2', 'mean')],
    sort_col_ascends=[True, False],
)

# inspect
print(full_df_latex)
full_df_format

#### Plot of model performance by eigenfunction {-}

In [ ]:
# plot R^2 or MSE results
import os 

# metric_col_key = 'R2|mean'
metric_col_key = 'mse|mean'
fig_save_path = os.path.join(
    "{PATH_TO_SAVE_DIR}",
    f"{metric_col_key}_plot"
)

# flatten multi-index cols
df_plot = df.copy()
df_plot.columns = df.columns.map('|'.join).str.strip('|')

# enforce floor R^2 of 0.
if 'R2' in metric_col_key:
    df_plot[df_plot[metric_col_key] < 0.] = 0.

# group single eigenvector results by model
df_plot = df_plot.reset_index() \
    .set_index('evec') \
    .groupby('model')[metric_col_key]

colors = [
    'tab:blue',
    'tab:green',
    'tab:orange',
    'tab:purple',
    'tab:brown',
    'tab:pink',
    'tab:gray',
    'tab:olive',
    'tab:cyan'
]

plt.rcParams.update({
    "figure.figsize": (10., 5.),
    "lines.linewidth": 2.2,
    "axes.prop_cycle": mpl.cycler(color=colors),
    "text.usetex": False, # True,
    "font.family": "serif",
    "font.sans-serif": ["Helvetica"],
    "font.size": 12
})
fig, ax = plt.subplots()
df_plot.plot(legend=True, ax=ax)

# reorder legend labels
rerun_final_modelnames = {
    "mcn_spectral-t=0.5, (32,16)": "MFCN-low-pass-spectral-0.5",
    "mcn_spectral-t=1.0, (32,16)": "MFCN-low-pass-spectral-1.0",
    "mfcn_spectral-t=0.5": "MFCN-wavelet-spectral-0.5",
}
final_modelnames = ru.PUB_MODELNAMES_DICT | rerun_final_modelnames
handles, labels = ax.get_legend_handles_labels()
labels = [final_modelnames[label] for label in labels] # ru.PUB_MODELNAMES_DICT[label]

'''optional: manually sort legend'''
# new_order_is = (4, 6, 5, 2, 7, 1, 0, 3)
# if len(new_order_is) != len(handles):
#     raise Exception("New sorting order indexes does not match handles!")
# handles = [handles[i] for i in new_order_is]
# labels = [labels[i] for i in new_order_is]

# final plot formatting
ax.set_xticks(range(2, 22))
ax.set_xticklabels([f"{i}" for i in range(2, 22)])
ax.set_xlabel("target eigenvector")
if 'R' in metric_col_key:
    ax.set_ylabel("$R^2$")
    ax.legend(
        handles=handles, 
        labels=labels, 
        loc='lower left',
        prop={'size': 9.5}, 
        bbox_to_anchor=(0.15, 0.45)
    )
elif 'mse' in metric_col_key:
    ax.set_ylabel("MSE")
    ax.legend(
        handles=handles,
        labels=labels, 
        loc='upper right',
        prop={'size': 10}, 
        bbox_to_anchor=(1.006, 1.015)
    )
    
plt.grid()
# optional: plot title
# plt.title(
#     "Ellipsoid node regression task:\n"
#     r"individual $L_n$ eigenvector value prediction 5-fold CV performance"
# )
plt.savefig(fig_save_path)
plt.show()